In [29]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [30]:
with open("data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [31]:
context_length = 8

x = enc_text[:context_length]
y = enc_text[1:context_length + 1]

print("Input tokens:", x)
print("Target tokens:   ", y)

Input tokens: [40, 367, 2885, 1464, 1807, 3619, 402, 271]
Target tokens:    [367, 2885, 1464, 1807, 3619, 402, 271, 10899]


In [32]:
import torch

print(torch.__version__)

2.8.0


In [33]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [34]:
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, enc_text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(enc_text, allowed_special={"<|endoftext|>"})
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_seq = token_ids[i:i + max_length]
            target_seq = token_ids[i + 1:i + max_length + 1]
            
            self.input_ids.append(torch.tensor(input_seq, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_seq, dtype=torch.long))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [35]:
def create_dataloader(txt, batch_size = 2, max_length = 256,
                      stride = 128, shuffle = True, drop_last = True,
                      num_workers = 0):
    
    tokenizer = tiktoken.get_encoding("gpt2")
    
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                            drop_last=drop_last, num_workers=num_workers)
    
    return dataloader



In [36]:
dataloaders = create_dataloader(raw_text, batch_size=1, max_length=4, stride=1)

dataloader_iter = iter(dataloaders)
first_batch = next(dataloader_iter)
print("Input batch:", first_batch)

Input batch: [tensor([[ 645, 1551, 1051,  286]]), tensor([[1551, 1051,  286, 1683]])]


In [37]:
second_batch = next(dataloader_iter)
print("Input batch:", second_batch)

Input batch: [tensor([[  11,  351,  257, 6487]]), tensor([[ 351,  257, 6487,  326]])]


In [38]:
dataloaders = create_dataloader(raw_text, batch_size=8, max_length=4, stride=4)

dataloader_iter = iter(dataloaders)
first_batch = next(dataloader_iter)
print("Input batch:", first_batch)

Input batch: [tensor([[  407, 17901,   683,   866],
        [  547,    11,   262, 15393],
        [22474,    62,  4964,   502],
        [28060,     6,   416,   617],
        [  351,   534,   670,  2474],
        [  422,   502,    13,  1320],
        [  198,   818,   262,  5391],
        [  996,   484,   547, 12548]]), tensor([[17901,   683,   866,    11],
        [   11,   262, 15393,   286],
        [   62,  4964,   502,    11],
        [    6,   416,   617,   530],
        [  534,   670,  2474,   198],
        [  502,    13,  1320, 41851],
        [  818,   262,  5391,    76],
        [  484,   547, 12548,   287]])]


In [39]:
vocab_size = tokenizer.n_vocab
output_dim = 256

torch.manual_seed(42)
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [40]:
token_embedding = token_embedding_layer(first_batch[0])
print("Token embedding shape:", token_embedding.shape)

Token embedding shape: torch.Size([8, 4, 256])


In [46]:
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

pos_embedding = pos_embedding_layer(torch.arange(context_length))

In [47]:
pos_embedding.shape

torch.Size([4, 256])

In [50]:
pos_embedding

tensor([[ 2.0634e-01, -3.2075e-02,  2.0340e-01,  ...,  7.1777e-01,
         -3.7598e-01,  5.4003e-01],
        [ 1.4862e-02, -3.8179e-01,  2.5129e-03,  ..., -9.5267e-01,
          1.4046e+00,  1.6583e+00],
        [-4.5511e-01, -5.9036e-01, -9.0341e-02,  ...,  9.5559e-01,
          5.2958e-01, -2.8266e-01],
        [ 2.9336e-01, -1.0375e+00, -2.7479e-01,  ...,  2.8263e+00,
          3.6057e-01, -1.1282e-01]], grad_fn=<EmbeddingBackward0>)

In [48]:
input_embedding = token_embedding + pos_embedding
print("Input embedding shape:", input_embedding.shape)

Input embedding shape: torch.Size([8, 4, 256])
